In [1]:
!pip -q install transformers datasets accelerate seqeval scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
!pip -q install -U transformers datasets accelerate seqeval scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.3 MB/s eta 0:00:00


In [4]:
!pip -q uninstall -y transformers
!pip -q install transformers==4.46.3 datasets accelerate seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 42.1 MB/s eta 0:00:00


In [1]:
import ast
import os
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple

# =====================
# 1) Install in Colab:
# !pip -q install transformers datasets accelerate seqeval scikit-learn
# =====================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

TRAIN_PATH = '/content/train_dataset.tsv'
TEST_PATH = '/content/private_test_dataset.csv'
OUTPUT_PATH = '/content/submission_transformer.csv'
MODEL_NAME = 'cointegrated/rubert-tiny2'
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 5
LR = 3e-5

train_df = pd.read_csv(TRAIN_PATH, sep='\t')
test_df = pd.read_csv(TEST_PATH)

def parse_target(x):
    return ast.literal_eval(x)

all_labels = sorted({span[2] for x in train_df['target'] for span in parse_target(x)})
label2id = {'O': 0}
for lab in all_labels:
    label2id[f'B-{lab}'] = len(label2id)
    label2id[f'I-{lab}'] = len(label2id)
id2label = {v: k for k, v in label2id.items()}


def spans_to_char_tags(text: str, spans: List[Tuple[int, int, str]]) -> List[str]:
    tags = ['O'] * len(text)
    for start, end, cat in spans:
        if start < 0 or end > len(text) or start >= end:
            continue
        tags[start] = f'B-{cat}'
        for i in range(start + 1, end):
            tags[i] = f'I-{cat}'
    return tags


def build_token_labels(text, spans, tokenizer, max_len=MAX_LEN):
    char_tags = spans_to_char_tags(text, spans)
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_len,
        padding='max_length'
    )
    labels = []
    prev_word_end = None
    for (start, end), input_id in zip(enc['offset_mapping'], enc['input_ids']):
        if start == end:
            labels.append(-100)
            continue
        token_tags = char_tags[start:end]
        non_o = [t for t in token_tags if t != 'O']
        if not non_o:
            labels.append(label2id['O'])
        else:
            # take first visible tag inside token span
            labels.append(label2id[non_o[0]])
    enc.pop('offset_mapping')
    enc['labels'] = labels
    return enc

from sklearn.model_selection import train_test_split
tr_df, va_df = train_test_split(train_df, test_size=0.15, random_state=SEED)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = [
    build_token_labels(row.text, parse_target(row.target), tokenizer)
    for row in tr_df.itertuples(index=False)
]
val_ds = [
    build_token_labels(row.text, parse_target(row.target), tokenizer)
    for row in va_df.itertuples(index=False)
]

from datasets import Dataset
train_ds = Dataset.from_list(train_ds)
val_ds = Dataset.from_list(val_ds)

from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


def decode_spans_from_logits(text, logits, offsets):
    pred_ids = logits.argmax(-1)
    spans = []
    current_start = None
    current_end = None
    current_cat = None
    for pred_id, (start, end) in zip(pred_ids, offsets):
        if start == end:
            continue
        tag = id2label[int(pred_id)]
        if tag == 'O':
            if current_start is not None:
                spans.append((current_start, current_end, current_cat))
                current_start = current_end = current_cat = None
            continue
        prefix, cat = tag.split('-', 1)
        if prefix == 'B' or current_cat != cat or current_start is None:
            if current_start is not None:
                spans.append((current_start, current_end, current_cat))
            current_start, current_end, current_cat = start, end, cat
        else:
            current_end = end
    if current_start is not None:
        spans.append((current_start, current_end, current_cat))
    # cleanup overlaps / invalid spans
    cleaned = []
    for s, e, c in sorted(spans):
        if s >= e:
            continue
        if cleaned and s < cleaned[-1][1]:
            if (e - s) > (cleaned[-1][1] - cleaned[-1][0]):
                cleaned[-1] = (s, e, c)
        else:
            cleaned.append((s, e, c))
    return cleaned


def exact_span_f1(eval_pred):
    logits, labels = eval_pred
    # rough token-level micro F1 proxy for model selection inside Trainer
    pred = logits.argmax(-1)
    mask = labels != -100
    y_true = labels[mask]
    y_pred = pred[mask]
    tp = int((y_true == y_pred).sum())
    precision = tp / max(len(y_pred), 1)
    recall = tp / max(len(y_true), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return {'token_micro_f1': f1}

from transformers import TrainingArguments

args = TrainingArguments(
    output_dir='/content/ner_runs',
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy='epoch',
    report_to='none',
    fp16=True
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# =====================
# Exact span validation after training
# =====================

def predict_one(text: str):
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LEN,
        padding='max_length',
        return_tensors='pt'
    )
    offsets = enc.pop('offset_mapping')[0].tolist()
    outputs = model(**{k: v.to(model.device) for k, v in enc.items()})
    logits = outputs.logits.detach().cpu().numpy()[0]
    return decode_spans_from_logits(text, logits, offsets)


def strict_micro_f1(df):
    tp = fp = fn = 0
    for row in df.itertuples(index=False):
        gold = set(parse_target(row.target))
        pred = set(predict_one(row.text))
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return precision, recall, f1

p, r, f1 = strict_micro_f1(va_df.reset_index(drop=True))
print({'strict_precision': p, 'strict_recall': r, 'strict_f1': f1})

# =====================
# Predict test and save submission
# =====================
preds = [str(predict_one(text)) for text in test_df['text']]
sub = pd.DataFrame({'id': test_df['id'], 'target': preds})
sub.to_csv(OUTPUT_PATH, index=False)
print('saved:', OUTPUT_PATH)
print(sub.head())


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NameError: name 'tokenized_ds' is not defined

In [2]:
# -*- coding: utf-8 -*-
"""
Baseline for Alfa case private_test_dataset.csv

Что делает:
1) Читает private_test_dataset.csv
2) Запускает локальную NER/LLM-модель
3) Ищет персональные данные в тексте
4) Формирует несколько вариантов submission:
   - submission_json.csv
   - submission_spans.csv
   - submission_parquet.parquet

ВАЖНО:
- Без sample_submission.csv невозможно гарантировать точный формат сабмита.
- Ниже используется локальная модель через HuggingFace, а не внешний API.
- Перед отправкой на Kaggle обязательно сверь названия колонок с sample_submission.
"""

import os
import re
import json
import ast
import math
import pandas as pd
from typing import List, Dict, Any, Tuple

# =========================
# CONFIG
# =========================

TEST_PATH = "private_test_dataset.csv"

# Модель можно заменить на любую локальную instruction / NER модель.
# Для русскоязычного baseline часто удобнее взять токен-классификацию или instruct LLM.
# Ниже код рассчитан на text-generation модель.
MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")

MAX_NEW_TOKENS = 256
BATCH_SIZE = 4
DEVICE_MAP = "auto"

# Набор категорий можно расширить под кейс
ALLOWED_LABELS = [
    "PERSON",
    "PHONE",
    "EMAIL",
    "PASSPORT",
    "SNILS",
    "INN",
    "CARD",
    "ACCOUNT",
    "ADDRESS",
    "DATE_OF_BIRTH",
    "LOGIN",
    "PASSWORD",
    "OTHER_PD"
]

# =========================
# UTILS
# =========================

def safe_json_loads(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_json_block(text: str):
    """
    Пытается вытащить JSON из ответа модели.
    """
    text = text.strip()

    # 1) прямой json
    obj = safe_json_loads(text)
    if obj is not None:
        return obj

    # 2) ```json ... ```
    m = re.search(r"```json\s*(.*?)\s*```", text, flags=re.S | re.I)
    if m:
        obj = safe_json_loads(m.group(1))
        if obj is not None:
            return obj

    # 3) первый {...} или [...]
    m = re.search(r"(\{.*\}|\[.*\])", text, flags=re.S)
    if m:
        candidate = m.group(1)
        obj = safe_json_loads(candidate)
        if obj is not None:
            return obj

    return None

def normalize_entities(raw_entities: Any, text: str) -> List[Dict[str, Any]]:
    """
    Приводит сущности к формату:
    [
      {
        "start": int,
        "end": int,
        "label": str,
        "text": str
      },
      ...
    ]
    """
    if raw_entities is None:
        return []

    if isinstance(raw_entities, dict) and "entities" in raw_entities:
        raw_entities = raw_entities["entities"]

    if not isinstance(raw_entities, list):
        return []

    result = []
    for ent in raw_entities:
        if not isinstance(ent, dict):
            continue

        start = ent.get("start")
        end = ent.get("end")
        label = ent.get("label") or ent.get("category") or ent.get("type")
        span_text = ent.get("text")

        if label is None:
            label = "OTHER_PD"
        label = str(label).strip().upper()

        if label not in ALLOWED_LABELS:
            label = "OTHER_PD"

        # если координат нет, попробуем найти по text
        if (start is None or end is None) and span_text:
            found = text.find(span_text)
            if found != -1:
                start = found
                end = found + len(span_text)

        # валидация координат
        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        span_text = text[start:end]

        result.append({
            "start": start,
            "end": end,
            "label": label,
            "text": span_text
        })

    # удаляем дубли
    uniq = {}
    for x in result:
        key = (x["start"], x["end"], x["label"])
        uniq[key] = x

    result = list(uniq.values())
    result.sort(key=lambda x: (x["start"], x["end"], x["label"]))
    return result

def regex_fallback(text: str) -> List[Dict[str, Any]]:
    """
    Фоллбек-эвристики, если модель не вернула валидный JSON.
    """
    patterns = [
        (r"\b[\w\.-]+@[\w\.-]+\.\w+\b", "EMAIL"),
        (r"(?:\+7|8)[\s\-()]?\d{3}[\s\-()]?\d{3}[\s\-()]?\d{2}[\s\-()]?\d{2}", "PHONE"),
        (r"\b\d{4}\s?\d{4}\s?\d{4}\s?\d{4}\b", "CARD"),
        (r"\b\d{3}-\d{3}-\d{3}\s\d{2}\b", "SNILS"),
        (r"\b\d{12}\b", "INN"),
        (r"\b\d{4}\s?\d{6}\b", "PASSPORT"),
    ]

    entities = []
    for pattern, label in patterns:
        for m in re.finditer(pattern, text):
            entities.append({
                "start": m.start(),
                "end": m.end(),
                "label": label,
                "text": m.group(0)
            })

    # Очень грубый поиск ФИО: три слова с заглавной буквы
    for m in re.finditer(r"\b[А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+\b", text):
        entities.append({
            "start": m.start(),
            "end": m.end(),
            "label": "PERSON",
            "text": m.group(0)
        })

    # дедуп
    uniq = {}
    for x in entities:
        uniq[(x["start"], x["end"], x["label"])] = x
    return list(sorted(uniq.values(), key=lambda x: (x["start"], x["end"], x["label"])))

# =========================
# MODEL
# =========================

def build_prompt(text: str) -> str:
    return f"""
Ты решаешь задачу NER по поиску персональных данных в русском тексте.

Нужно вернуть ТОЛЬКО JSON формата:
{{
  "entities": [
    {{
      "start": 0,
      "end": 10,
      "label": "PHONE"
    }}
  ]
}}

Правила:
- start и end — символьные индексы в исходном тексте
- end не включается
- Возвращай только найденные сущности
- Если сущностей нет, верни {{"entities": []}}
- Никакого поясняющего текста
- Допустимые label:
  {ALLOWED_LABELS}

Текст:
{text}
""".strip()

def load_generator():
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        device_map=DEVICE_MAP
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )
    return pipe

def predict_entities(pipe, text: str) -> List[Dict[str, Any]]:
    prompt = build_prompt(text)

    try:
        out = pipe(
            prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            return_full_text=False,
        )
        generated = out[0]["generated_text"]
        parsed = extract_json_block(generated)
        ents = normalize_entities(parsed, text)
        if ents:
            return ents
    except Exception as e:
        print(f"[WARN] model failed: {e}")

    return regex_fallback(text)

# =========================
# MAIN
# =========================

def main():
    df = pd.read_csv(TEST_PATH)
    assert len(df) == 3552, f"Ожидалось 3552 строк, получено {len(df)}"

    print("Loaded:", df.shape)
    print(df.head())

    pipe = load_generator()

    predictions = []
    for i, row in df.iterrows():
        text = str(row["text"])
        entities = predict_entities(pipe, text)

        predictions.append({
            "id": row["id"],
            "text": text,
            "entities": entities
        })

        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(df)}")

    pred_df = pd.DataFrame(predictions)

    # =========================
    # ВАРИАНТ 1: json в одной колонке
    # =========================
    sub_json = pred_df[["id"]].copy()
    sub_json["prediction"] = pred_df["entities"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )
    sub_json.to_csv("submission_json.csv", index=False)

    # =========================
    # ВАРИАНТ 2: spans/categories отдельно
    # =========================
    def to_spans(entities):
        return json.dumps([[e["start"], e["end"]] for e in entities], ensure_ascii=False)

    def to_labels(entities):
        return json.dumps([e["label"] for e in entities], ensure_ascii=False)

    sub_spans = pred_df[["id"]].copy()
    sub_spans["spans"] = pred_df["entities"].apply(to_spans)
    sub_spans["labels"] = pred_df["entities"].apply(to_labels)
    sub_spans.to_csv("submission_spans.csv", index=False)

    # =========================
    # ВАРИАНТ 3: parquet
    # =========================
    sub_parquet = sub_json.copy()
    sub_parquet.to_parquet("submission_parquet.parquet", index=False)

    print("\nFiles saved:")
    print(" - submission_json.csv")
    print(" - submission_spans.csv")
    print(" - submission_parquet.parquet")

    print("\nPreview submission_json.csv:")
    print(sub_json.head())

if __name__ == "__main__":
    main()


Loaded: (3552, 2)
   id                                               text
0   0  Банкомат не запрашивает CVV-код. Для снятия на...
1   1  Уточните, пожалуйста, ФИО получателя и точную ...
2   2  Проверьте папку «Спам» в вашей электронной поч...
3   3  Я пытаюсь обновить данные в личном кабинете, н...
4   4  Хочу подключить СМС-информирование. Нужно ли у...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set.

Processed 50/3552


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

KeyboardInterrupt: 

In [3]:
# -*- coding: utf-8 -*-
import os
import re
import json
import time
import pandas as pd
import torch
from typing import List, Dict, Any
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================

TEST_PATH = "private_test_dataset.csv"
MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")

MAX_NEW_TOKENS = 256
BATCH_SIZE = 8   # для T4 пробуй 4, 6 или 8
DEVICE_MAP = "auto"

ALLOWED_LABELS = [
    "PERSON",
    "PHONE",
    "EMAIL",
    "PASSPORT",
    "SNILS",
    "INN",
    "CARD",
    "ACCOUNT",
    "ADDRESS",
    "DATE_OF_BIRTH",
    "LOGIN",
    "PASSWORD",
    "OTHER_PD"
]

# =========================
# UTILS
# =========================

def safe_json_loads(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_json_block(text: str):
    text = text.strip()

    obj = safe_json_loads(text)
    if obj is not None:
        return obj

    m = re.search(r"```json\s*(.*?)\s*```", text, flags=re.S | re.I)
    if m:
        obj = safe_json_loads(m.group(1))
        if obj is not None:
            return obj

    m = re.search(r"(\{.*\}|\[.*\])", text, flags=re.S)
    if m:
        obj = safe_json_loads(m.group(1))
        if obj is not None:
            return obj

    return None

def normalize_entities(raw_entities: Any, text: str) -> List[Dict[str, Any]]:
    if raw_entities is None:
        return []

    if isinstance(raw_entities, dict) and "entities" in raw_entities:
        raw_entities = raw_entities["entities"]

    if not isinstance(raw_entities, list):
        return []

    result = []
    for ent in raw_entities:
        if not isinstance(ent, dict):
            continue

        start = ent.get("start")
        end = ent.get("end")
        label = ent.get("label") or ent.get("category") or ent.get("type")
        span_text = ent.get("text")

        if label is None:
            label = "OTHER_PD"
        label = str(label).strip().upper()

        if label not in ALLOWED_LABELS:
            label = "OTHER_PD"

        if (start is None or end is None) and span_text:
            found = text.find(span_text)
            if found != -1:
                start = found
                end = found + len(span_text)

        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        result.append({
            "start": start,
            "end": end,
            "label": label,
            "text": text[start:end]
        })

    uniq = {}
    for x in result:
        uniq[(x["start"], x["end"], x["label"])] = x

    result = list(uniq.values())
    result.sort(key=lambda x: (x["start"], x["end"], x["label"]))
    return result

def regex_fallback(text: str) -> List[Dict[str, Any]]:
    patterns = [
        (r"\b[\w\.-]+@[\w\.-]+\.\w+\b", "EMAIL"),
        (r"(?:\+7|8)[\s\-()]?\d{3}[\s\-()]?\d{3}[\s\-()]?\d{2}[\s\-()]?\d{2}", "PHONE"),
        (r"\b\d{4}\s?\d{4}\s?\d{4}\s?\d{4}\b", "CARD"),
        (r"\b\d{3}-\d{3}-\d{3}\s\d{2}\b", "SNILS"),
        (r"\b\d{12}\b", "INN"),
        (r"\b\d{4}\s?\d{6}\b", "PASSPORT"),
    ]

    entities = []
    for pattern, label in patterns:
        for m in re.finditer(pattern, text):
            entities.append({
                "start": m.start(),
                "end": m.end(),
                "label": label,
                "text": m.group(0)
            })

    for m in re.finditer(r"\b[А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+\b", text):
        entities.append({
            "start": m.start(),
            "end": m.end(),
            "label": "PERSON",
            "text": m.group(0)
        })

    uniq = {}
    for x in entities:
        uniq[(x["start"], x["end"], x["label"])] = x

    return list(sorted(uniq.values(), key=lambda x: (x["start"], x["end"], x["label"])))

# =========================
# MODEL
# =========================

def build_prompt(text: str) -> str:
    return f"""
Ты решаешь задачу NER по поиску персональных данных в русском тексте.

Нужно вернуть ТОЛЬКО JSON формата:
{{
  "entities": [
    {{
      "start": 0,
      "end": 10,
      "label": "PHONE"
    }}
  ]
}}

Правила:
- start и end — символьные индексы в исходном тексте
- end не включается
- Возвращай только найденные сущности
- Если сущностей нет, верни {{"entities": []}}
- Никакого поясняющего текста
- Допустимые label:
  {ALLOWED_LABELS}

Текст:
{text}
""".strip()

def load_generator():
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        device_map=DEVICE_MAP,
        torch_dtype=torch.float16
    )

    # убираем конфликт max_length из generation_config
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.max_length = None
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        pad_token_id=tokenizer.pad_token_id,
    )
    return pipe

def predict_entities_batch(pipe, texts: List[str]) -> List[List[Dict[str, Any]]]:
    prompts = [build_prompt(t) for t in texts]

    outputs = pipe(
        prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        batch_size=BATCH_SIZE,
    )

    all_entities = []
    for text, out in zip(texts, outputs):
        try:
            # pipeline для batch обычно возвращает list[list[dict]]
            if isinstance(out, list):
                generated = out[0]["generated_text"]
            else:
                generated = out["generated_text"]

            parsed = extract_json_block(generated)
            ents = normalize_entities(parsed, text)
            if not ents:
                ents = regex_fallback(text)
        except Exception:
            ents = regex_fallback(text)

        all_entities.append(ents)

    return all_entities

# =========================
# MAIN
# =========================

def batched(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]

def main():
    df = pd.read_csv(TEST_PATH)
    assert len(df) == 3552, f"Ожидалось 3552 строк, получено {len(df)}"

    print("Loaded:", df.shape)
    print(df.head())

    pipe = load_generator()

    ids = df["id"].tolist()
    texts = df["text"].astype(str).tolist()

    predictions = []
    start_all = time.time()

    for batch_idx, batch_texts in enumerate(tqdm(list(batched(texts, BATCH_SIZE)), desc="Inference")):
        batch_start = batch_idx * BATCH_SIZE
        batch_ids = ids[batch_start: batch_start + len(batch_texts)]

        batch_entities = predict_entities_batch(pipe, batch_texts)

        for row_id, text, entities in zip(batch_ids, batch_texts, batch_entities):
            predictions.append({
                "id": row_id,
                "text": text,
                "entities": entities
            })

        processed = len(predictions)
        elapsed = time.time() - start_all
        avg = elapsed / processed
        eta_min = avg * (len(df) - processed) / 60
        print(f"Processed {processed}/{len(df)} | avg={avg:.2f}s per row | ETA={eta_min:.1f} min")

    pred_df = pd.DataFrame(predictions)

    sub_json = pred_df[["id"]].copy()
    sub_json["prediction"] = pred_df["entities"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )
    sub_json.to_csv("submission_json.csv", index=False)

    def to_spans(entities):
        return json.dumps([[e["start"], e["end"]] for e in entities], ensure_ascii=False)

    def to_labels(entities):
        return json.dumps([e["label"] for e in entities], ensure_ascii=False)

    sub_spans = pred_df[["id"]].copy()
    sub_spans["spans"] = pred_df["entities"].apply(to_spans)
    sub_spans["labels"] = pred_df["entities"].apply(to_labels)
    sub_spans.to_csv("submission_spans.csv", index=False)

    sub_json.to_parquet("submission_parquet.parquet", index=False)

    print("\nFiles saved:")
    print(" - submission_json.csv")
    print(" - submission_spans.csv")
    print(" - submission_parquet.parquet")
    print("\nPreview submission_json.csv:")
    print(sub_json.head())

if __name__ == "__main__":
    main()


Loaded: (3552, 2)
   id                                               text
0   0  Банкомат не запрашивает CVV-код. Для снятия на...
1   1  Уточните, пожалуйста, ФИО получателя и точную ...
2   2  Проверьте папку «Спам» в вашей электронной поч...
3   3  Я пытаюсь обновить данные в личном кабинете, н...
4   4  Хочу подключить СМС-информирование. Нужно ли у...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Inference:   0%|          | 0/444 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 363.81 MiB is free. Including non-PyTorch memory, this process has 14.21 GiB memory in use. Of the allocated memory 14.05 GiB is allocated by PyTorch, and 27.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)